In [4]:
import pandas as pd 

In [5]:
booking_requests_df = pd.read_csv("cleaned_data/booking_requests.csv")
cleaners_df = pd.read_csv("cleaned_data/cleaners.csv")
customers_df = pd.read_csv("cleaned_data/customers.csv")
marketing_spend_df = pd.read_csv("cleaned_data/marketing_spend.csv")

# 1. Channels

In [11]:
# add the customer's acquisition channel to each booking
bookings_df = booking_requests_df.merge(customers_df[["customer_id", "acquisition_channel"]], on="customer_id")
bookings_df

,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month,acquisition_channel
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0,True,False,2025-01-01,Google Ads
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0,True,False,2025-05-01,Google Ads
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN,False,False,2025-02-01,Meta Ads
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0,True,False,2026-01-01,Google Ads
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0,True,False,2025-07-01,Google Ads
...,...,...,...,...,...,...,...,...,...,...,...,...,...
132159,R0038441,C004736,Lyon,2025-10-24,4,108.0,fulfilled,P00234,5.0,True,False,2025-10-01,Organic
132160,R0040942,C009999,Bordeaux,2025-10-25,3,81.0,fulfilled,P00170,5.0,True,False,2025-10-01,Organic
132161,R0027337,C002467,Paris,2025-09-21,2,54.0,fulfilled,P00501,5.0,True,False,2025-09-01,Google Ads
132162,R0106418,C010209,Paris,2026-05-20,3,81.0,fulfilled,P01096,5.0,True,False,2026-05-01,Google Ads


In [ ]:
# 1. Raw comparison by channel
customers_per_channel = customers_df.groupby("acquisition_channel").size()

raw_df = bookings_df.groupby("acquisition_channel").agg(
    bookings=("request_id", "size"),
    fulfilment_rate=("fulfilled", "mean"),
    cancel_rate=("canc", "mean"),
    avg_rating=("rating", "mean"),
)
raw_df["bookings_per_customer"] = raw_df["bookings"] / customers_per_channel
print(raw_df.round(3))

                     bookings  fulfilment_rate  cancel_rate  avg_rating  \
acquisition_channel                                                       
Google Ads              45586            0.825        0.050       4.259   
Meta Ads                39952            0.820        0.049       4.264   
Organic                 32785            0.823        0.049       4.255   
Referral                13841            0.819        0.049       4.271   

                     bookings_per_customer  
acquisition_channel                         
Google Ads                           5.307  
Meta Ads                             5.427  
Organic                              5.308  
Referral                             5.528  


In [13]:
# 2. Cost per customer (only paid channels have spend)
demand_spend = (marketing_spend_df[marketing_spend_df["side"] == "demand"]
                .groupby("channel")["spend_eur"].sum())
print((demand_spend / customers_per_channel).round(1).rename("CAC"))

channel
Google Ads    23.9
Meta Ads      18.8
Organic        NaN
Referral       NaN
Name: CAC, dtype: float64


In [14]:
# 3. Like with like: same comparison inside each city
print(bookings_df.pivot_table(index="city", columns="acquisition_channel",
                              values="fulfilled", aggfunc="mean").round(3))
print(bookings_df.pivot_table(index="city", columns="acquisition_channel",
                              values="rating", aggfunc="mean").round(3))

acquisition_channel  Google Ads  Meta Ads  Organic  Referral
city                                                        
Bordeaux                  0.908     0.904    0.902     0.903
Lille                     0.904     0.908    0.888     0.895
Lyon                      0.616     0.608    0.611     0.611
Paris                     0.901     0.904    0.904     0.903
acquisition_channel  Google Ads  Meta Ads  Organic  Referral
city                                                        
Bordeaux                  4.255     4.270    4.261     4.275
Lille                     4.236     4.288    4.238     4.283
Lyon                      4.273     4.261    4.268     4.268
Paris                     4.258     4.261    4.251     4.269


# 2. City

In [20]:
# 1. Raw comparison by city
print(booking_requests_df.groupby("city").agg(
    requests=("request_id", "size"),
    fulfilment_rate=("fulfilled", "mean"),
    cancel_rate=("canc", "mean"),
    avg_rating=("rating", "mean"),
    avg_price=("price_eur", "mean"),
).round(3))

          requests  fulfilment_rate  cancel_rate  avg_rating  avg_price
city                                                                   
Bordeaux     17166            0.905        0.050       4.263     72.846
Lille         8181            0.900        0.053       4.256     73.086
Lyon         36590            0.612        0.048       4.267     72.752
Paris        70227            0.903        0.049       4.258     73.024


Lyon’s fulfilment rate is 61%, against about 90% in the other three cities.

In [21]:
# 2. Like with like: same comparison within each month and within each job length
print(booking_requests_df.pivot_table(index="month", columns="city", values="fulfilled", aggfunc="mean").round(2))
print(booking_requests_df.pivot_table(index="hours", columns="city", values="fulfilled", aggfunc="mean").round(3))

city        Bordeaux  Lille  Lyon  Paris
month                                   
2025-01-01       NaN    NaN  0.39   0.90
2025-02-01       NaN    NaN  0.74   0.89
2025-03-01       NaN    NaN  0.49   0.86
2025-04-01      0.94    NaN  0.60   0.86
2025-05-01      0.90    NaN  0.65   0.91
2025-06-01      0.92    NaN  0.69   0.93
2025-07-01      0.93    NaN  0.72   0.91
2025-08-01      0.89    NaN  0.67   0.89
2025-09-01      0.88   0.91  0.63   0.89
2025-10-01      0.89   0.96  0.62   0.88
2025-11-01      0.94   0.92  0.64   0.90
2025-12-01      0.88   0.85  0.66   0.89
2026-01-01      0.93   0.86  0.61   0.93
2026-02-01      0.89   0.93  0.62   0.90
2026-03-01      0.93   0.88  0.62   0.91
2026-04-01      0.89   0.89  0.55   0.93
2026-05-01      0.90   0.94  0.58   0.91
2026-06-01      0.90   0.88  0.55   0.89
city   Bordeaux  Lille   Lyon  Paris
hours                               
2         0.903  0.893  0.613  0.903
3         0.909  0.908  0.608  0.903
4         0.899  0.899  0.620  0

In [22]:
# 3. Why Lyon? Requests per active cleaner in the last month (June 2026)
june_requests = booking_requests_df[booking_requests_df["month"] == "2026-06-01"].groupby("city").size()
active_cleaners = cleaners_df[
    (cleaners_df["onboarded_date"] <= "2026-06-30")
    & (cleaners_df["deactivated_date"].isna() | (cleaners_df["deactivated_date"] >= "2026-06-01"))
].groupby("city").size()
print(pd.DataFrame({"requests": june_requests, "active_cleaners": active_cleaners,
                    "requests_per_cleaner": june_requests / active_cleaners}).round(1))

          requests  active_cleaners  requests_per_cleaner
city                                                     
Bordeaux      1969              143                  13.8
Lille         1425               76                  18.8
Lyon          3756               77                  48.8
Paris         6855              298                  23.0


Lyon has about 49 requests per cleaner, against 23 in Paris, 19 in Lille and 14 in Bordeaux. So Lyon’s extra requests go unserved.

In [23]:
# 4. Share of marketing spend going to recruiting cleaners
spend_df = marketing_spend_df.groupby(["city", "side"])["spend_eur"].sum().unstack()
print((spend_df["supply"] / (spend_df["supply"] + spend_df["demand"])).round(2).rename("supply_share"))

city
Bordeaux    0.44
Lille       0.27
Lyon        0.11
Paris       0.22
Name: supply_share, dtype: float64


# 3. Period

In [25]:
# 1. Whole business, month by month
monthly_df = booking_requests_df.groupby("month").agg(
    requests=("request_id", "size"),
    fulfilment_rate=("fulfilled", "mean"),
    cancel_rate=("canc", "mean"),
    avg_rating=("rating", "mean"),
)
monthly_df["new_customers"] = customers_df.groupby("month").size()
demand_spend = marketing_spend_df[marketing_spend_df["side"] == "demand"].groupby("month")["spend_eur"].sum()
monthly_df["cac"] = demand_spend / monthly_df["new_customers"]
print(monthly_df.round(3))

            requests  fulfilment_rate  cancel_rate  avg_rating  new_customers  \
month                                                                           
2025-01-01       668            0.725        0.039       4.198            NaN   
2025-02-01      1419            0.850        0.049       4.258            NaN   
2025-03-01      2194            0.743        0.048       4.235            NaN   
2025-04-01      2818            0.790        0.047       4.240            NaN   
2025-05-01      3652            0.837        0.052       4.213            NaN   
2025-06-01      4456            0.856        0.042       4.271            NaN   
2025-07-01      5203            0.859        0.052       4.242            NaN   
2025-08-01      5997            0.826        0.050       4.295            NaN   
2025-09-01      7029            0.816        0.049       4.249            NaN   
2025-10-01      7792            0.812        0.050       4.260            NaN   
2025-11-01      8546        

In [ ]:
# 2. Like with like: fulfilment month by month, Lyon vs the other cities
booking_requests_df["group"] = booking_requests_df["city"].where(booking_requests_df["city"] == "Lyon", "Other cities")
print(booking_requests_df.pivot_table(index="month", columns="group", values="fulfilled", aggfunc="mean").round(3))

group        Lyon  Other cities
month                          
2025-01-01  0.395         0.901
2025-02-01  0.744         0.891
2025-03-01  0.486         0.861
2025-04-01  0.604         0.867
2025-05-01  0.652         0.912
2025-06-01  0.692         0.924
2025-07-01  0.718         0.912
2025-08-01  0.669         0.889
2025-09-01  0.625         0.890
2025-10-01  0.619         0.886
2025-11-01  0.641         0.911
2025-12-01  0.658         0.886
2026-01-01  0.612         0.923
2026-02-01  0.618         0.900
2026-03-01  0.619         0.909
2026-04-01  0.552         0.921
2026-05-01  0.580         0.915
2026-06-01  0.548         0.889


In [ ]:
# 3. Day of week (0 = Monday)

booking_requests_df["weekday"] = booking_requests_df["request_date"].dt.dayofweek
#print(booking_requests_df.groupby("weekday").agg(requests=("request_id", "size"),fulfilment_rate=("fulfilled", "mean"),cancel_rate=("canc", "mean"),).round(3))

AttributeError: Can only use .dt accessor with datetimelike values